# G3-VPR: Notebook Độc Lập 100% Cho Kaggle & Google Colab (Không Cần GitHub!)
**Cách chạy trên Kaggle chỉ với 3 thao tác:**
1. Ở cột **Settings** bên phải: chọn **Accelerator -> `GPU T4 x2`** (hoặc `P100`) và bật **Internet -> `ON`**.
2. Ở mục **Input** bên phải: bấm **`+ Add Input`** -> gắn **`amaralibey/gsv-cities`**.
3. Bấm **`Run All`** trên thanh công cụ! *(Toàn bộ file `.py` của dự án đã được nhúng sẵn tự giải nén ở Cell 1 — không cần đưa lên GitHub hay tải thêm code).* 

In [ ]:
# Cell 1: Tu dong giai nen toan bo code G3-VPR ngay trong Kaggle/Colab (KHONG CAN GITHUB!)
import base64, io, os, tarfile, torch

WORK_ROOT = "/kaggle/working/D2VPR" if os.path.isdir("/kaggle/working") else os.path.abspath("./D2VPR_cloud")
os.makedirs(WORK_ROOT, exist_ok=True)

ARCHIVE_B64 = (
    "..."
    "f/af/Wf/2X/2n9/68/8BSYZPZgDIBQA="
)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(ARCHIVE_B64)), mode="r:gz") as tf:
    tf.extractall(WORK_ROOT)
os.chdir(WORK_ROOT)
print("[OK] Da giai nen toan bo du an tai:", os.getcwd())
print("[OK] Cac file chinh:", sorted([f for f in os.listdir(".") if f.endswith(".py")]))
print("PyTorch:", torch.__version__, "| CUDA GPU:", torch.cuda.is_available(), "| So GPU:", torch.cuda.device_count())
if not torch.cuda.is_available():
    print("CANH BAO: Ban chua bat GPU! Vao Settings ben phai -> Accelerator -> chon GPU T4 x2 va bat Internet ON!")
else:
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB)")

!pip uninstall -y -q torchaudio
!pip install -q --no-deps pytorch-metric-learning && pip install -q faiss-cpu

In [ ]:
# Cell 2: Tai trong so nen DINOv2 ViT-S/14 & Tu dong tim duong dan GSV-Cities
import os, urllib.request
if os.path.isdir("/kaggle/working/D2VPR"):
    os.chdir("/kaggle/working/D2VPR")

dinov2_ckpt = "./dinov2_vits14_pretrain.pth"
if not os.path.isfile(dinov2_ckpt):
    print("Dang tai trong so DINOv2 ViT-S/14...")
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth",
        dinov2_ckpt
    )
print("[OK] DINOv2-S san sang:", os.path.isfile(dinov2_ckpt))

GSV_PATH = None
if os.path.isdir("/kaggle/input"):
    for r, dirs, _ in os.walk("/kaggle/input"):
        if "Dataframes" in dirs and "Images" in dirs:
            GSV_PATH = r
            break
if GSV_PATH is None:
    import kagglehub
    GSV_PATH = kagglehub.dataset_download("amaralibey/gsv-cities")
print("[OK] Da tim thay GSV-Cities tai:", GSV_PATH)

In [ ]:
# Cell 3: Kiem tra nhanh kien truc G3-VPR, tinh bat bien BS=1 & toc do GPU
if os.path.isdir("/kaggle/working/D2VPR"): os.chdir("/kaggle/working/D2VPR")
!python -u experiments/exp2_architecture_verify_and_profile.py

In [ ]:
# Cell 4: Huan luyen Mo hinh Chinh G3-VPR (M5_full) voi 2 Teachers (MegaLoc 2D + Depth-Anything-V2 3D)
# Mac dinh max_places=20000 (~80,000 anh/epoch) de chay gon trong 1 phien Kaggle T4.
# Neu muon train full 100% 62,000 dia diem, doi --max_places 0.
import os
if os.path.isdir("/kaggle/working/D2VPR"): os.chdir("/kaggle/working/D2VPR")
SAVE_DIR = "./checkpoints/g3vpr_M5_full"
RESUME_ARG = f"--resume_ckpt {SAVE_DIR}/g3vpr_last.pth" if os.path.isfile(f"{SAVE_DIR}/g3vpr_last.pth") else ""

cmd = (
    f"python -u train_g3vpr.py "
    f"--ablation M5_full "
    f"--gsv_path \"{GSV_PATH}\" "
    f"--dinov2_path ./dinov2_vits14_pretrain.pth "
    f"{RESUME_ARG} "
    f"--teacher_2d megaloc "
    f"--teacher_3d depth_anything_v2 "
    f"--bs 32 --grad_accum 2 --epochs 10 --max_places 20000 --lr 2e-4 --devices 1 "
    f"--save_dir \"{SAVE_DIR}\""
)
print("Running:", cmd)
!yes | {cmd}

In [ ]:
# Cell 5: Chay Bang Ablation Study (Table 4: M1 -> M2 -> M3 -> M4 -> M5_full)
if os.path.isdir("/kaggle/working/D2VPR"): os.chdir("/kaggle/working/D2VPR")
ABLATION_CITIES = "London,Boston,Melbourne,Chicago,Toronto"
for abl in ["M1", "M2", "M3", "M4", "M5_full"]:
    out_d = f"./checkpoints/ablation_{abl}"
    !yes | python -u train_g3vpr.py --ablation {abl} --gsv_path "{GSV_PATH}" --cities "{ABLATION_CITIES}" \
        --dinov2_path ./dinov2_vits14_pretrain.pth --teacher_2d megaloc --teacher_3d depth_anything_v2 \
        --bs 32 --grad_accum 2 --epochs 5 --max_places 6000 --devices 1 --save_dir "{out_d}"

In [ ]:
# Cell 6: Danh gia BS=1 Matryoshka (128D..4096D), ve Hinh 4 & Nen file ZIP ket qua tai ve may!
import os, shutil
if os.path.isdir("/kaggle/working/D2VPR"): os.chdir("/kaggle/working/D2VPR")
!python -u eval_g3vpr_sota.py --ckpt "./checkpoints/g3vpr_M5_full/g3vpr_universal_best.pth" --query_bs 1 --output_json ./results/g3vpr_sota_eval.json
!python -u experiments/visualize_gaussians.py --ckpt "./checkpoints/g3vpr_M5_full/g3vpr_universal_best.pth"

zip_out = "/kaggle/working/g3vpr_paper_artifacts" if os.path.isdir("/kaggle/working") else "./g3vpr_paper_artifacts"
os.makedirs("./export_bundle/results", exist_ok=True)
os.makedirs("./export_bundle/checkpoints", exist_ok=True)
if os.path.isdir("./results"):
    shutil.copytree("./results", "./export_bundle/results", dirs_exist_ok=True)
best_p = "./checkpoints/g3vpr_M5_full/g3vpr_universal_best.pth"
if os.path.isfile(best_p):
    shutil.copy2(best_p, "./export_bundle/checkpoints/g3vpr_universal_best.pth")
for root, _, files in os.walk("./checkpoints"):
    for fn in files:
        if fn.endswith(".json") or fn.endswith(".csv"):
            rel_d = os.path.relpath(root, "./checkpoints")
            os.makedirs(f"./export_bundle/checkpoints/{rel_d}", exist_ok=True)
            shutil.copy2(os.path.join(root, fn), f"./export_bundle/checkpoints/{rel_d}/{fn}")
archive_file = shutil.make_archive(zip_out, "zip", "./export_bundle")
print("[HOAN TAT] Tai file ket qua tai muc Output ben phai:", archive_file)